In [ ]:
# Run first: locate the repository root from any notebook subfolder.
from pathlib import Path
import os
import sys
project_root = next((candidate for candidate in [Path.cwd(), *Path.cwd().parents]
                     if (candidate / "app.py").is_file()
                     and (candidate / "retrieval_final.py").is_file()), None)
if project_root is None:
    raise RuntimeError("Start Jupyter within the project repository.")
os.chdir(project_root)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))


# Full-corpus TF-IDF evaluation

This notebook evaluates the final TF-IDF retriever by itself on all cleaned Enron emails and the same 250 grounded questions. It reuses the saved TF-IDF vectorizer and sparse matrix when they already exist.

In [ ]:
# Import the libraries
import time
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from scipy.sparse import load_npz, save_npz
from sklearn.feature_extraction.text import TfidfVectorizer

EMAILS_FILE = Path('enron_clean.csv')
EVALUATION_FILE = Path('data/enron_evaluation_250.csv')
TFIDF_VECTORIZER_FILE = Path('full_corpus_tfidf_vectorizer.joblib')
TFIDF_MATRIX_FILE = Path('full_corpus_tfidf_matrix.npz')
TOP_K_VALUES = [1, 3, 5, 10, 50, 100]

In [ ]:
# Load the emails and questions, validate their columns, and locate every labelled source email.
emails = pd.read_csv(EMAILS_FILE)
evaluation_questions = pd.read_csv(EVALUATION_FILE)
required_email_columns = {'email_id', 'sender', 'recipients', 'subject', 'body'}
required_question_columns = {'question', 'answer', 'email_id', 'difficulty', 'retrieval_type'}
missing_email_columns = required_email_columns - set(emails.columns)
missing_question_columns = required_question_columns - set(evaluation_questions.columns)
if missing_email_columns or missing_question_columns:
    raise ValueError(
        f'Missing email columns: {sorted(missing_email_columns)}; '
        f'missing question columns: {sorted(missing_question_columns)}'
    )
emails['email_id'] = emails['email_id'].astype(str)
evaluation_questions['email_id'] = evaluation_questions['email_id'].astype(str)
email_ids = emails['email_id'].to_numpy()
email_position_by_id = {email_id: position for position, email_id in enumerate(email_ids)}
missing_sources = set(evaluation_questions['email_id']) - set(email_position_by_id)
if missing_sources:
    raise ValueError(f'{len(missing_sources)} labelled source emails are missing.')
target_positions = np.asarray(
    [email_position_by_id[email_id] for email_id in evaluation_questions['email_id']],
    dtype=np.int64,
)
print(f'Loaded {len(emails):,} emails and {len(evaluation_questions):,} questions.')

In [ ]:
# Confirm the existing easy, medium, and hard evaluation groups before retrieval.
difficulty_counts = (
    evaluation_questions['difficulty'].value_counts()
    .reindex(['easy', 'medium', 'hard'])
    .rename_axis('difficulty')
    .reset_index(name='questions')
)
display(difficulty_counts)

In [ ]:
# Build the sparse TF-IDF email space once, or load the files created by the hybrid evaluation.
def build_search_text(email_frame):
    return (
        email_frame['sender'].fillna('') + ' ' +
        email_frame['recipients'].fillna('') + ' ' +
        email_frame['subject'].fillna('') + ' ' +
        email_frame['body'].fillna('')
    )

started = time.perf_counter()
if TFIDF_VECTORIZER_FILE.exists() and TFIDF_MATRIX_FILE.exists():
    tfidf_vectorizer = joblib.load(TFIDF_VECTORIZER_FILE)
    email_tfidf_matrix = load_npz(TFIDF_MATRIX_FILE)
else:
    tfidf_vectorizer = TfidfVectorizer(
        stop_words='english', lowercase=True, ngram_range=(1, 2),
        sublinear_tf=True, dtype=np.float32,
    )
    email_tfidf_matrix = tfidf_vectorizer.fit_transform(build_search_text(emails))
    joblib.dump(tfidf_vectorizer, TFIDF_VECTORIZER_FILE, compress=3)
    save_npz(TFIDF_MATRIX_FILE, email_tfidf_matrix, compressed=True)
if email_tfidf_matrix.shape[0] != len(emails):
    raise ValueError('The TF-IDF matrix does not match enron_clean.csv.')
print(f'TF-IDF matrix: {email_tfidf_matrix.shape}')
print(f'Ready in {time.perf_counter() - started:.1f} seconds.')

In [ ]:
# Score all questions together and calculate the exact rank of every labelled email.
retrieval_started = time.perf_counter()
question_matrix = tfidf_vectorizer.transform(evaluation_questions['question'].astype(str))
all_scores = (question_matrix @ email_tfidf_matrix.T).toarray()
relevant_ranks = []
for question_position, scores in enumerate(all_scores):
    target_score = scores[target_positions[question_position]]
    relevant_ranks.append(
        1 + int(np.sum(scores > target_score)) if target_score > 0 else np.inf
    )
relevant_ranks = np.asarray(relevant_ranks, dtype=float)
evaluation_questions['tfidf_rank'] = relevant_ranks
print(f'Evaluated all questions in {time.perf_counter() - retrieval_started:.1f} seconds.')

In [ ]:
# Show overall accuracy and the requested easy, medium, and hard breakdown.
def metric_row(label, ranks, label_column='method'):
    row = {label_column: label, 'questions': len(ranks)}
    for k in TOP_K_VALUES:
        row[f'recall_at_{k}'] = float(np.mean(ranks <= k))
    row['mrr'] = float(np.mean(1.0 / ranks))
    return row

overall_results = pd.DataFrame([metric_row('TF-IDF', relevant_ranks)])
difficulty_results = pd.DataFrame([
    metric_row(difficulty, group['tfidf_rank'].to_numpy(dtype=float), 'difficulty')
    for difficulty, group in evaluation_questions.groupby('difficulty', sort=False)
])
formatters = {**{f'recall_at_{k}': '{:.1%}' for k in TOP_K_VALUES}, 'mrr': '{:.3f}'}
display(overall_results.style.format(formatters))
display(difficulty_results.style.format(formatters).background_gradient(
    subset=[f'recall_at_{k}' for k in TOP_K_VALUES] + ['mrr'], cmap='YlGn'
))

In [ ]:
# Plot Recall@10 for the easy, medium, and hard question groups.
plt.figure(figsize=(11, 5))
sns.barplot(data=difficulty_results, x='difficulty', y='recall_at_10', order=['easy', 'medium', 'hard'], color='#4C78A8')
plt.ylim(0, 1)
plt.xlabel('Question difficulty')
plt.ylabel('Recall@10')
plt.title('Full-corpus TF-IDF Recall@10 by difficulty')
plt.tight_layout()
plt.show()